# 📊 Akshare + 本地环境：沪深300完整因子 → 回测 → 可视化

**环境**：本地Python环境  
**股票池**：沪深300（CSI300）  
**流程**：因子构造 → 回测 → 可视化分析

## 0️⃣ 环境准备

In [6]:
# 安装必要的依赖包
!pip install -q akshare pandas numpy matplotlib seaborn tqdm statsmodels


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import akshare as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

print(f'Akshare版本: {ak.__version__}')

Akshare版本: 1.18.4


## 1️⃣ 获取沪深300成分股数据

In [8]:
# 获取沪深300成分股列表
csi300_stocks = ak.index_stock_cons('000300.SH')
print(f'沪深300成分股数量: {len(csi300_stocks)}')
print('前10只成分股:')
csi300_stocks.head(10)

AttributeError: 'NoneType' object has no attribute 'find'

In [ ]:
# 选择一个示例股票进行分析
symbol = '600519'  # 贵州茅台
stock_name = csi300_stocks[csi300_stocks['成分券代码'] == symbol]['成分券名称'].values[0]
print(f'示例股票: {stock_name} ({symbol})')

## 2️⃣ 获取历史行情数据

In [ ]:
# 设置时间范围
end_date = datetime.now().strftime('%Y%m%d')
start_date = (datetime.now() - timedelta(days=365*3)).strftime('%Y%m%d')  # 近3年数据

# 获取股票历史行情数据
stock_data = ak.stock_zh_a_hist(symbol=symbol, start_date=start_date, end_date=end_date, adjust='qfq')

# 数据预处理
stock_data['date'] = pd.to_datetime(stock_data['日期'])
stock_data.set_index('date', inplace=True)
stock_data = stock_data.sort_index()

print(f'数据时间范围: {stock_data.index.min()} 至 {stock_data.index.max()}')
print(f'数据形状: {stock_data.shape}')
stock_data.tail()

## 3️⃣ 因子构造

In [ ]:
# 计算常用因子
df = stock_data.copy()

# 1. 收益率因子
df['return'] = df['收盘'].pct_change()

# 2. 波动率因子（20日）
df['volatility'] = df['return'].rolling(20).std() * np.sqrt(252)

# 3. MACD因子
exp1 = df['收盘'].ewm(span=12, adjust=False).mean()
exp2 = df['收盘'].ewm(span=26, adjust=False).mean()
df['macd'] = exp1 - exp2
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

# 4. RSI因子（14日）
delta = df['收盘'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['rsi'] = 100 - (100 / (1 + rs))

# 5. 移动平均线因子
df['ma5'] = df['收盘'].rolling(5).mean()
df['ma10'] = df['收盘'].rolling(10).mean()
df['ma20'] = df['收盘'].rolling(20).mean()
df['ma60'] = df['收盘'].rolling(60).mean()

# 6. 成交量因子
df['volume_ma5'] = df['成交量'].rolling(5).mean()
df['volume_ma20'] = df['成交量'].rolling(20).mean()
df['volume_ratio'] = df['成交量'] / df['volume_ma20']

# 查看因子数据
df.tail()

## 4️⃣ 策略回测

In [ ]:
# 简单的MACD策略回测
df_strategy = df.copy()

# 生成信号
df_strategy['signal'] = 0
df_strategy.loc[df_strategy['macd'] > df_strategy['macd_signal'], 'signal'] = 1
df_strategy.loc[df_strategy['macd'] < df_strategy['macd_signal'], 'signal'] = -1

# 计算策略收益率
df_strategy['strategy_return'] = df_strategy['signal'].shift(1) * df_strategy['return']

# 计算累计收益率
df_strategy['cumulative_return'] = (1 + df_strategy['return']).cumprod()
df_strategy['cumulative_strategy_return'] = (1 + df_strategy['strategy_return']).cumprod()

print(f'基准收益率: {((df_strategy['cumulative_return'].iloc[-1] - 1) * 100):.2f}%')
print(f'策略收益率: {((df_strategy['cumulative_strategy_return'].iloc[-1] - 1) * 100):.2f}%')

## 5️⃣ 可视化分析

In [ ]:
# 绘制累计收益率曲线
plt.figure(figsize=(12, 6))
plt.plot(df_strategy.index, df_strategy['cumulative_return'], label='基准收益率', linewidth=2)
plt.plot(df_strategy.index, df_strategy['cumulative_strategy_return'], label='策略收益率', linewidth=2)
plt.title(f'{stock_name} ({symbol}) 策略回测结果', fontsize=16)
plt.xlabel('日期', fontsize=12)
plt.ylabel('累计收益率', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 绘制K线图和MACD指标
import mplfinance as mpf

# 准备K线数据
ohlc_data = df[['开盘', '最高', '最低', '收盘', '成交量']].copy()
ohlc_data.columns = ['Open', 'High', 'Low', 'Close', 'Volume']

# 创建MACD子图
apds = [mpf.make_addplot(df['macd'], panel=1, color='blue', title='MACD'),
        mpf.make_addplot(df['macd_signal'], panel=1, color='orange'),
        mpf.make_addplot(df['macd_hist'], panel=1, type='bar', color='gray')]

# 绘制K线图
mpf.plot(ohlc_data.loc['2025':],
         type='candle',
         volume=True,
         addplot=apds,
         title=f'{stock_name} ({symbol}) K线图',
         style='yahoo',
         figsize=(14, 8))

## 6️⃣ 扩展：批量分析沪深300成分股

In [ ]:
# 批量分析前10只沪深300成分股
top_10_stocks = csi300_stocks.head(10)
results = []

for _, row in top_10_stocks.iterrows():
    symbol = row['成分券代码']
    name = row['成分券名称']
    
    try:
        # 获取数据
        stock_data = ak.stock_zh_a_hist(symbol=symbol, start_date=start_date, end_date=end_date, adjust='qfq')
        stock_data['date'] = pd.to_datetime(stock_data['日期'])
        stock_data.set_index('date', inplace=True)
        stock_data = stock_data.sort_index()
        
        # 计算收益率
        stock_data['return'] = stock_data['收盘'].pct_change()
        total_return = (stock_data['收盘'].iloc[-1] / stock_data['收盘'].iloc[0] - 1) * 100
        
        # 计算波动率
        volatility = stock_data['return'].std() * np.sqrt(252) * 100
        
        results.append({
            '代码': symbol,
            '名称': name,
            '总收益率(%)': total_return,
            '波动率(%)': volatility
        })
        
        print(f'处理完成: {name} ({symbol})')
        
    except Exception as e:
        print(f'处理失败: {name} ({symbol}) - {str(e)}')
        continue

# 展示结果
results_df = pd.DataFrame(results)
results_df.sort_values('总收益率(%)', ascending=False, inplace=True)
results_df

In [ ]:
# 可视化前10只股票的收益率和波动率
plt.figure(figsize=(14, 6))

# 绘制收益率条形图
plt.subplot(1, 2, 1)
sns.barplot(x='名称', y='总收益率(%)', data=results_df)
plt.title('沪深300成分股收益率', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

# 绘制波动率条形图
plt.subplot(1, 2, 2)
sns.barplot(x='名称', y='波动率(%)', data=results_df)
plt.title('沪深300成分股波动率', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()